In [0]:

# SmartCity AI - Bronze initialization

CATALOG = "workspace"
SCHEMA = "smartcity_bronze"
VOLUME = "smartcity_raw"

# Create our Bronze database
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}"
)

# Create a managed landing volume
spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}"
)

RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

print("SmartCity AI storage ready!")
print("Raw files location:", RAW_PATH)


SmartCity AI storage ready!
Raw files location: /Volumes/workspace/smartcity_bronze/smartcity_raw


In [0]:

# SmartCity AI - Verify uploaded datasets

RAW_PATH = "/Volumes/workspace/smartcity_bronze/smartcity_raw"

expected_files = {
    "yellow_tripdata_2025-01.parquet",
    "taxi_zone_lookup.csv",
    "dot_traffic_speeds_2025-01_sample.csv",
    "nyc_crashes_2025-01.csv"
}

files = dbutils.fs.ls(RAW_PATH)

print("SMARTCITY AI - RAW DATA VERIFICATION")
print("-" * 45)

for file in files:
    print(f"{file.name}: {file.size / (1024**2):.2f} MB")

uploaded = {file.name for file in files}
missing = expected_files - uploaded

assert not missing, f"Missing files: {missing}"

print("\nSUCCESS: ALL 4 DATASETS VERIFIED!")


SMARTCITY AI - RAW DATA VERIFICATION
---------------------------------------------
dot_traffic_speeds_2025-01_sample.csv: 2.24 MB
nyc_crashes_2025-01.csv: 1.67 MB
taxi_zone_lookup.csv: 0.01 MB
yellow_tripdata_2025-01.parquet: 56.42 MB

SUCCESS: ALL 4 DATASETS VERIFIED!


In [0]:

# SMARTCITY AI — BRONZE DELTA INGESTION

from pyspark.sql.functions import current_timestamp, lit

CATALOG = "workspace"
SCHEMA = "smartcity_bronze"
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/smartcity_raw"

FILES = {
    "taxi_trips": "yellow_tripdata_2025-01.parquet",
    "taxi_zones": "taxi_zone_lookup.csv",
    "traffic_speeds": "dot_traffic_speeds_2025-01_sample.csv",
    "road_collisions": "nyc_crashes_2025-01.csv"
}

TABLES = {}

for name, filename in FILES.items():

    path = f"{RAW_PATH}/{filename}"
    table_name = f"{CATALOG}.{SCHEMA}.{name}"

    print(f"Processing: {filename}")

    # Read original file
    if filename.endswith(".parquet"):
        df = spark.read.parquet(path)
    else:
        df = (
            spark.read.format("csv")
            .option("header", "true")
            .option("inferSchema", "false")
            .load(path)
        )

    # Add ingestion metadata
    df = (
        df.withColumn("_source_file", lit(filename))
          .withColumn("_ingested_at", current_timestamp())
    )

    # Write managed Delta table
    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(table_name)
    )

    TABLES[name] = table_name

    print(f"SUCCESS: {table_name}")

print("\nALL FOUR BRONZE TABLES CREATED!")


Processing: yellow_tripdata_2025-01.parquet
SUCCESS: workspace.smartcity_bronze.taxi_trips
Processing: taxi_zone_lookup.csv
SUCCESS: workspace.smartcity_bronze.taxi_zones
Processing: dot_traffic_speeds_2025-01_sample.csv
SUCCESS: workspace.smartcity_bronze.traffic_speeds
Processing: nyc_crashes_2025-01.csv
SUCCESS: workspace.smartcity_bronze.road_collisions

ALL FOUR BRONZE TABLES CREATED!


In [0]:

# SMARTCITY AI — FINAL BRONZE VALIDATION

tables = [
    "taxi_zones",
    "taxi_trips",
    "traffic_speeds",
    "road_collisions"
]

print("SMARTCITY AI - BRONZE QUALITY REPORT")
print("=" * 55)

for name in tables:
    full_name = f"workspace.smartcity_bronze.{name}"

    # Verify table exists
    assert spark.catalog.tableExists(full_name), (
        f"Missing table: {name}"
    )

    df = spark.table(full_name)

    rows = df.count()
    columns = len(df.columns)

    # Verify Delta format
    info = spark.sql(
        f"DESCRIBE DETAIL {full_name}"
    ).first().asDict()

    format_name = info["format"]

    print(f"\nTABLE: {name}")
    print(f"Rows: {rows:,}")
    print(f"Columns: {columns}")
    print(f"Format: {format_name}")

    assert rows > 0, f"Empty table: {name}"
    assert format_name.lower() == "delta"

    print("STATUS: PASSED")

print("\n" + "=" * 55)
print("SUCCESS: ALL 4 BRONZE TABLES VERIFIED!")


SMARTCITY AI - BRONZE QUALITY REPORT

TABLE: taxi_zones
Rows: 265
Columns: 6
Format: delta
STATUS: PASSED

TABLE: taxi_trips
Rows: 3,475,226
Columns: 22
Format: delta
STATUS: PASSED

TABLE: traffic_speeds
Rows: 5,000
Columns: 15
Format: delta
STATUS: PASSED

TABLE: road_collisions
Rows: 19,290
Columns: 31
Format: delta
STATUS: PASSED

SUCCESS: ALL 4 BRONZE TABLES VERIFIED!


In [0]:

# SMARTCITY AI - BRONZE ENGINEERING REPORT

tables = [
    "taxi_trips",
    "taxi_zones",
    "traffic_speeds",
    "road_collisions"
]

print("# SmartCity AI - Bronze Validation Report")
print()
print("| Table | Rows | Columns | Format |")
print("|---|---:|---:|---|")

for name in tables:
    full_name = f"workspace.smartcity_bronze.{name}"
    df = spark.table(full_name)

    rows = df.count()
    cols = len(df.columns)

    fmt = spark.sql(
        f"DESCRIBE DETAIL {full_name}"
    ).select("format").first()[0]

    print(f"| {name} | {rows:,} | {cols} | {fmt} |")

print()
print("Validation: All four tables available.")
print("Storage: Unity Catalog managed Delta tables.")


# SmartCity AI - Bronze Validation Report

| Table | Rows | Columns | Format |
|---|---:|---:|---|
| taxi_trips | 3,475,226 | 22 | delta |
| taxi_zones | 265 | 6 | delta |
| traffic_speeds | 5,000 | 15 | delta |
| road_collisions | 19,290 | 31 | delta |

Validation: All four tables available.
Storage: Unity Catalog managed Delta tables.
